In [18]:
import pandas as pd
from pathlib import Path
from datetime import date
from pyproj import Geod

# generating 5ºx5º tiles

In [19]:
# load data from https://craters.sjrdesign.net/
CSV_FILE = "/Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY/Catalog_Mars_Release_2020_1kmPlus_FullMorphData.csv"
df = pd.read_csv(CSV_FILE, engine="python")

# force the three essential numeric columns
num_cols = ["DIAM_CIRC_IMG", "LAT_CIRC_IMG", "LON_CIRC_IMG"]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

# assign each crater to a 5 °×5 ° tile (lower-left corner)
df["tile_lat"] = (df["LAT_CIRC_IMG"] // 5 * 5).astype(int)
df["tile_lon"] = (df["LON_CIRC_IMG"] // 5 * 5).astype(int)


OUT_DIR = Path("tiles_scc") # where *.scc files go

# IAU-2000 Mars reference ellipsoid (km)
A_EQUAT = 3396.19
B_POLAR = 3376.20
GEOD = Geod(a=A_EQUAT*1000, b=B_POLAR*1000)   # metres -> pyproj

In [20]:
# helper functions

def tile_boundary(lat0: int, lon0: int):
    '''Returns the five (lon,lat) vertices that close the square tile.'''
    return [
        (lon0,     lat0),
        (lon0+5,   lat0),
        (lon0+5,   lat0+5),
        (lon0,     lat0+5),
        (lon0,     lat0) # to close the polygon
    ]

def tile_area_km2(lat0: int, lon0: int) -> float:
    '''Exact ellipsoidal area of the 5°×5° tile in km².'''
    lons, lats = zip(*tile_boundary(lat0, lon0)[:-1]) # four corners only
    area_m2, _ = GEOD.polygon_area_perimeter(lons, lats)
    return abs(area_m2) / 1_000_000


In [21]:
# Writing the *.scc files

OUT_DIR.mkdir(exist_ok=True)
today = date.today().strftime("%d.%m.%Y")

for (lat0, lon0), tile in df.groupby(["tile_lat", "tile_lon"]):
    area = tile_area_km2(lat0, lon0)
    fname = OUT_DIR / f"tile_lat{lat0:+03d}_lon{lon0:+04d}.scc"

    # ---- header ---------------------------------------------------
    header = [
        "# Spatial crater count",
        "#",
        f"# Date of measurement = {today}",
        "#",
        "# Ellipsoid axes:",
        f"a_axis_radius = {A_EQUAT:.2f} <km>",
        f"b_axis_radius = {A_EQUAT:.2f} <km>",
        f"c_axis_radius = {A_EQUAT:.2f} <km>",
        "coordinate_system_name = unknown",
        "#",
        "# area_shapes:",
        "unit_boundary = {vertex,sub_area,tag,lon,lat",
        "#",
        f"# Area_name1 = tile {lat0}/{lon0}",
        f"# Area_size1 = {area:.11f} <km^2>",
        "#",
    ]

    # ---- polygon vertices ----------------------------------------
    bnd = [
        f"{i+1}\t1\text\t{lon:.12f}\t{lat:.12f}"
        for i, (lon, lat) in enumerate(tile_boundary(lat0, lon0))
    ] + ["}"]          # closes unit_boundary

    # ---- crater diameters section --------------------------------
    crater_block = [
        "#",
        f"Total_area = {area:.11f} <km^2>",
        "#",
        "# crater_diameters:",
        "crater = {diam,lon,lat"
    ]

    crater_rows = (
        tile[["DIAM_CIRC_IMG", "LON_CIRC_IMG", "LAT_CIRC_IMG"]]
        .round(6)
        .astype(str)
        .agg("\t".join, axis=1)
        .tolist()
        + ["}"]        # closes crater list
    )

    # ---- write file ----------------------------------------------
    fname.write_text(
        "\n".join(header + bnd + crater_block + crater_rows),
        encoding="utf-8"
    )

    print(f"✓ {fname.name:35s}  ({len(tile):4d} craters,  area {area:8.1f} km²)")

print("\nAll tiles written to", OUT_DIR.resolve())

✓ tile_lat-90_lon+000.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+005.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+045.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+095.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+145.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+150.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+225.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+235.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+245.scc              (   2 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+250.scc              (   2 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+255.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+270.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+345.scc              (   2 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+355.scc             

## References

https://craters.sjrdesign.net/

https://home.csulb.edu/~rodrigue/geog441541/lectures/midterm/marsinspace.html#:~:text=Equatorial%20radius:%203%2C396%20km%20(Earth,km3%20(Earth:%201%2C083%2C200%2C000%2C000%20km

https://en.wikipedia.org/wiki/Spherical_cap

https://arxiv.org/pdf/1109.4448